# Copilot Credit Consumption — Ingester (Fabric)

Lands the **Power Platform message-consumption / entitlement reports** (`EntitlementConsumption*_MCSMessages_*.csv`) into Lakehouse Delta tables that the **AI Business Value Dashboard — 1905 Extra** PBIP reads for *finance-grade* credit reporting.

Why this exists: the transcript parser already derives an *in-product* `displayedCost` per conversation (the diagnostic "where did generative cost happen" view). These CSVs are the *billed* view — prepaid vs pay-as-you-go, per agent / per user / per environment. The two reconcile; this notebook brings in the billing side **without a manual import step** (the CSVs are landed into `Files/` by the companion Power Automate flow — see `3. Fabric/flows/`).

```
Power Platform Admin Centre  ─(export)─▶  Power Automate flow  ─▶  Lakehouse Files/credit_consumption/*.csv  ─▶  THIS notebook  ─▶  Delta
```

| Source CSV (prefix) | → Delta table | Grain |
| --- | --- | --- |
| `EntitlementConsumptionTenantDetailsReport_MCSMessages` | `dbo.credit_consumption_tenant` | environment × capacity × usage date |
| `EntitlementConsumptionTenantPerAgentDetailsReport_MCSMessages` | `dbo.credit_consumption_agent` | agent × feature × environment |
| `EntitlementConsumptionTenantPerUserDetailsReport_MCSMessages` | `dbo.credit_consumption_user` | user × agent |

**Graceful by design** — every report is *optional*. A customer who never sends consumption CSVs simply gets empty tables, and the PBIP's `Enable_Consumption` toggle keeps those pages dormant. Missing files are skipped with a warning, never an error.

**Multi-environment** — the PerAgent / Tenant reports already carry `Environment Id` / `Environment Name`, so a customer running 1 or 12 environments lands in the *same* tables with the environment as a column. Drop one combined export, or several per-environment exports into `Files/credit_consumption/` — the glob picks them all up and a `SourceFile` column preserves lineage.

**No app registration / Graph permission required** — this notebook only reads files already in the Lakehouse.

## 1. Configuration

Tag this cell as the pipeline **`parameters`** cell (cell ⋯ menu → *Toggle parameter cell*) so `CopilotAdoptionPipeline` can override `WRITE_MODE` per run. Output tables are schema-qualified (`dbo.`) to work with both schema-enabled and legacy Lakehouses.

In [ ]:
# === CONFIG ===  (tag this cell as the pipeline `parameters` cell)

# Folder in the attached Lakehouse where the Power Automate flow lands the CSVs.
SOURCE_DIR  = 'Files/credit_consumption'

# Filename globs -> target Delta table. The default Microsoft export names are matched
# case-insensitively and tolerate the trailing _30 / _180 / (1) day-window + dedupe suffixes.
REPORTS = {
    'dbo.credit_consumption_tenant': 'EntitlementConsumptionTenantDetailsReport_MCSMessages*',
    'dbo.credit_consumption_agent':  'EntitlementConsumptionTenantPerAgentDetailsReport_MCSMessages*',
    'dbo.credit_consumption_user':   'EntitlementConsumptionTenantPerUserDetailsReport_MCSMessages*',
}

WRITE_MODE  = 'overwrite'   # 'overwrite' = full snapshot (these reports are already a rolling window);
                            # 'append'    = keep history (a LoadDate column is always added either way)

ADD_LINEAGE = True          # add SourceFile + LoadDate columns to every row
STRICT      = False         # False = skip a missing/empty report with a warning; True = raise

## 2. Helpers

`_list_matches` resolves a glob against the Lakehouse `Files/` area (via `notebookutils.fs.ls` (mount-independent)). `_read_csv` parses with multiline + quote-escape so the reports' free-text columns (knowledge sources, scenario names) survive. `_sanitize` makes column names Delta-safe while the PBIP keeps reading the original names through Delta column-mapping.

In [ ]:
import os, glob, re, datetime
from pyspark.sql import functions as F

_LOAD_TS = datetime.datetime.now(datetime.timezone.utc).strftime('%Y-%m-%dT%H:%M:%SZ')

# Local POSIX mount for the attached Lakehouse: 'Files/x' -> '/lakehouse/default/Files/x'
def _local(path):
    return path if path.startswith('/') else f'/lakehouse/default/{path}'

def _list_matches(pattern):
    rx = re.compile('^' + re.escape(pattern).replace('\\*', '.*') + '$', re.IGNORECASE)
    try:
        entries = notebookutils.fs.ls(SOURCE_DIR)
    except Exception:
        return []
    hits = [e.name for e in entries if (not e.isDir) and rx.match(e.name) and e.name.lower().endswith('.csv')]
    return [f'{SOURCE_DIR}/{f}' for f in sorted(hits)]

def _read_csv(path):
    return (spark.read
            .option('header', True)
            .option('multiLine', True)
            .option('escape', '"')
            .option('encoding', 'UTF-8')
            .csv(path))

_INVALID = re.compile(r'[ ,;{}()\n\t=/-]')
def _sanitize(df):
    return df.toDF(*[_INVALID.sub('_', c.lstrip('\ufeff')) for c in df.columns])

print(f'Load timestamp: {_LOAD_TS}')
print(f'Source folder : {SOURCE_DIR}  (exists: {os.path.isdir(_local(SOURCE_DIR))})')

## 3. Ingest each report → Delta

Loops the `REPORTS` map. For each target table it unions every matching CSV (so per-environment exports merge into one table), adds lineage columns, sanitises, and writes. A missing report writes an **empty, correctly-named** table so the PBIP query never errors with *"the column … wasn't found"*.

In [ ]:
from pyspark.sql.types import StructType, StructField, StringType

# Canonical (sanitised) schemas — used to build an empty table when a report is absent,
# so downstream PBIP columns always resolve.
EMPTY_SCHEMAS = {
    'dbo.credit_consumption_tenant': ['BillingPlan_Id','BillingPlan_Name','Environment_Id','Environment_Name',
        'Capacity_Type','Entitled_Quantity','Prepaid_Consumed_Quantity','Pay_as_you_go_Consumed_Quantity','Usage_Date'],
    'dbo.credit_consumption_agent': ['Agent_Name','Agent_Id','Product','AI_Feature_Billable_Feature','Billed_credit',
        'Non_billed_credit','Channel','Knowledge_Sources','Tool_Used','LLM_Model','Scenario_Name','Environment_Id','Environment_Name'],
    'dbo.credit_consumption_user': ['User_Id','User_Email','Agent_Id','Agent_Name','Billable_credit_used',
        'Credits_used','M365_Copilot_Licensed'],
}

def _empty(cols):
    cols = list(cols)
    if ADD_LINEAGE:
        cols = cols + ['SourceFile', 'LoadDate']
    return spark.createDataFrame([], StructType([StructField(c, StringType(), True) for c in cols]))

def _write(df, table):
    (df.write.mode(WRITE_MODE)
        .option('overwriteSchema', 'true')
        .option('delta.columnMapping.mode', 'name')
        .option('delta.minReaderVersion', '2')
        .option('delta.minWriterVersion', '5')
        .format('delta').saveAsTable(table))

summary = []
for table, pattern in REPORTS.items():
    matches = _list_matches(pattern)
    if not matches:
        msg = f'no file matched "{pattern}"'
        if STRICT:
            raise FileNotFoundError(f'{table}: {msg}')
        print(f'⚠  {table:34} — {msg}; writing EMPTY table so the PBIP still resolves.')
        _write(_empty(EMPTY_SCHEMAS[table]), table)
        summary.append((table, 0, 0))
        continue

    frames = []
    for path in matches:
        d = _sanitize(_read_csv(path))
        if ADD_LINEAGE:
            d = d.withColumn('SourceFile', F.lit(os.path.basename(path))) \
                 .withColumn('LoadDate',   F.lit(_LOAD_TS))
        frames.append(d)

    # Union by name so per-environment exports with identical schemas merge cleanly.
    df = frames[0]
    for d in frames[1:]:
        df = df.unionByName(d, allowMissingColumns=True)

    n = df.count()
    _write(df, table)
    summary.append((table, len(matches), n))
    print(f'✓  {table:34} — {len(matches)} file(s), {n:,} rows -> written ({WRITE_MODE})')

print('\nDone. Credit-consumption Delta tables written. Refresh the PBIP / Direct Lake model to pick them up.')

## 4. Verify + quick reconciliation

Prints headline totals and — if the transcript table `agent_sessions` exists — the gap between transcript-native `displayedCost` and the billed credits, so you can sanity-check the two views side by side.

In [ ]:
def _num(col):
    return F.sum(F.coalesce(F.col(col).cast('double'), F.lit(0.0)))

for table, files, rows in summary:
    print(f'{table:34} files={files} rows={rows:,}')

try:
    agent = spark.table('dbo.credit_consumption_agent')
    if agent.count():
        tot = agent.select(_num('Billed_credit').alias('billed'),
                           _num('Non_billed_credit').alias('nonbilled')).collect()[0]
        print(f"\nBilling view  — billed credits: {tot['billed']:,.0f}   non-billed: {tot['nonbilled']:,.0f}")
except Exception as e:
    print('agent consumption summary skipped:', e)

try:
    sess = spark.table('dbo.agent_sessions')
    disp = sess.select(_num('total_displayed_cost').alias('disp')).collect()[0]['disp']
    print(f"Transcript view — displayedCost credits: {disp:,.0f}")
    print('(Expected to be LOWER than billed total: displayedCost only covers generative Dynamic-Plan steps.)')
except Exception:
    print('(agent_sessions not present yet — run the transcript parser to enable reconciliation.)')

---
**Connect the PBIP**: these three tables feed the semantic-model queries `Credit Consumption (Agent)`, `Credit Consumption (User)`, `Credit Consumption (Tenant)`, all gated by the `Enable_Consumption` parameter. Leave `Enable_Consumption = false` for customers who don't supply the reports and the billing visuals stay dormant — the transcript-native `Total Cost Units` measure keeps working regardless.